Test ROS2 Bridge before use python editor inside Isaac Sim:

In [ ]:
import rclpy
from rclpy.node import Node
rclpy.init()
print("ROS2 Bridge OK")

In [ ]:
from pxr import Usd, UsdGeom, UsdPhysics, Gf
import omni.usd

# Create a stage
omni.usd.get_context().new_stage()
stage = omni.usd.get_context().get_stage()

# Define the root Xform (transformable object)
rootxform = UsdGeom.Xform.Define(stage, "/World")

cubePath = "/World/cube"

# Properties of the geometry
cubePosition = Gf.Vec3f(0.0, 0.0, 0.0)
cubeOrientation = Gf.Quatf(1.0)
cubeScale = Gf.Vec3f(1.0, 1.0, 1.0)
cubeSize = 1.0 #unit is: meter
cubeHalfExtent = cubeSize/2.0

cubeGeom = UsdGeom.Cube.Define(stage, cubePath)
cubeGeom.CreateSizeAttr(cubeSize)
cubeGeom.CreateExtentAttr([Gf.Vec3f(-cubeHalfExtent), Gf.Vec3f(cubeHalfExtent)])
cubeGeom.AddTranslateOp().Set(cubePosition)
cubeGeom.AddOrientOp().Set(cubeOrientation)
cubeGeom.AddScaleOp().Set(cubeScale)
cubePrim = cubeGeom.GetPrim()
UsdPhysics.CollisionAPI.Apply(cubePrim)

# stage.RemovePrim(cubePath)

In [ ]:
import omni.kit.commands
import omni.usd
from pxr import UsdPhysics, PhysxSchema, UsdGeom, Gf

stage = omni.usd.get_context().get_stage()

# 导入 OBJ
omni.kit.commands.execute(
    "CreateReferenceCommand",
    usd_context=omni.usd.get_context(),
    path_to="/World/BackyardMesh",
    asset_path="/root/scene_clean.obj",
    instanceable=False)

mesh_prim = stage.GetPrimAtPath("/World/BackyardMesh")

# 坐标系对齐（COLMAP Y-up → Isaac Sim Z-up）
xform = UsdGeom.Xformable(mesh_prim)
xform.ClearXformOpOrder()
xform.AddRotateXYZOp().Set(Gf.Vec3f(-90, 0, 0))
xform.AddScaleOp().Set(Gf.Vec3f(1.0, 1.0, 1.0))  # 按需调整尺度

# 添加碰撞属性
UsdPhysics.CollisionAPI.Apply(mesh_prim)
mesh_col = UsdPhysics.MeshCollisionAPI.Apply(mesh_prim)
mesh_col.GetApproximationAttr().Set("sdf")  # 最精确，保留凹面

# 设为静态（不受重力）
rb = UsdPhysics.RigidBodyAPI.Apply(mesh_prim)
rb.GetRigidBodyEnabledAttr().Set(False)

print("Scene imported with SDF collision.")

In [ ]:
import carb
from omni.kit.scripting import BehaviorScript


class AddCollisionToObj(BehaviorScript):
    def on_init(self):
        carb.log_info(f"{type(self).__name__}.on_init()->{self.prim_path}")

    def on_destroy(self):
        carb.log_info(f"{type(self).__name__}.on_destroy()->{self.prim_path}")

    def on_play(self):
        carb.log_info(f"{type(self).__name__}.on_play()->{self.prim_path}")

    def on_pause(self):
        carb.log_info(f"{type(self).__name__}.on_pause()->{self.prim_path}")

    def on_stop(self):
        carb.log_info(f"{type(self).__name__}.on_stop()->{self.prim_path}")

    def on_update(self, current_time: float, delta_time: float):
        carb.log_info(f"{type(self).__name__}.on_update({current_time}, {delta_time})->{self.prim_path}")
